In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("data/hospital data analysis.csv", delimiter=",")

In [3]:
df.head(5)

,Patient_ID,Age,Gender,Condition,Procedure,Cost,Length_of_Stay,Readmission,Outcome,Satisfaction
0,1,45,Female,Heart Disease,Angioplasty,15000,5,No,Recovered,4
1,2,60,Male,Diabetes,Insulin Therapy,2000,3,Yes,Stable,3
2,3,32,Female,Fractured Arm,X-Ray and Splint,500,1,No,Recovered,5
3,4,75,Male,Stroke,CT Scan and Medication,10000,7,Yes,Stable,2
4,5,50,Female,Cancer,Surgery and Chemotherapy,25000,10,No,Recovered,4


In [4]:
df.tail(5)

,Patient_ID,Age,Gender,Condition,Procedure,Cost,Length_of_Stay,Readmission,Outcome,Satisfaction
979,996,68,Male,Hypertension,Medication and Counseling,1000,70,No,Stable,4
980,997,45,Female,Appendicitis,Appendectomy,8000,72,No,Recovered,3
981,998,40,Male,Fractured Leg,Cast and Physical Therapy,3000,72,No,Recovered,4
982,999,78,Female,Heart Attack,Cardiac Catheterization,18000,74,Yes,Stable,2
983,1000,25,Male,Allergic Reaction,Epinephrine Injection,100,68,No,Recovered,5


In [5]:
# Get basic stats
df.describe()

,Patient_ID,Age,Cost,Length_of_Stay,Satisfaction
count,984.000000,984.000000,984.000000,984.000000,984.000000
mean,500.329268,53.754065,8367.479675,37.663618,3.598577
std,288.979531,14.941135,7761.990976,19.595805,0.883002
min,1.000000,25.000000,100.000000,1.000000,2.000000
25%,250.750000,45.000000,1000.000000,21.000000,3.000000
50%,500.500000,55.000000,6000.000000,38.000000,4.000000
75%,750.250000,65.000000,15000.000000,54.250000,4.000000
max,1000.000000,78.000000,25000.000000,76.000000,5.000000


In [6]:
# check for missing values
df.isna().sum()

Patient_ID        0
Age               0
Gender            0
Condition         0
Procedure         0
Cost              0
Length_of_Stay    0
Readmission       0
Outcome           0
Satisfaction      0
dtype: int64

In [7]:
df.duplicated().any()

np.False_

In [8]:
# validate data

def validate_date(df):
    valid_rows = []
    invalid_rows = []

    for _, row in df.iterrows():
        if row["Age"] < 0 or row["Age"] > 120:
            invalid_rows.append(row)
            continue

        if row["Cost"] < 0:
            invalid_rows.append(row)
            continue

        if row["Length_of_Stay"] < 0:
            invalid_rows.append(row)
            continue

        if row["Satisfaction"] < 0 or row["Satisfaction"] > 5:
            invalid_rows.append(row)
            continue

        valid_rows.append(row)

    valid_df = pd.DataFrame(valid_rows)
    invalid_df = pd.DataFrame(invalid_rows)

    return valid_df, invalid_df

In [9]:
valid_df, invalid_df = validate_date(df)

In [10]:
valid_df.head(5)

,Patient_ID,Age,Gender,Condition,Procedure,Cost,Length_of_Stay,Readmission,Outcome,Satisfaction
0,1,45,Female,Heart Disease,Angioplasty,15000,5,No,Recovered,4
1,2,60,Male,Diabetes,Insulin Therapy,2000,3,Yes,Stable,3
2,3,32,Female,Fractured Arm,X-Ray and Splint,500,1,No,Recovered,5
3,4,75,Male,Stroke,CT Scan and Medication,10000,7,Yes,Stable,2
4,5,50,Female,Cancer,Surgery and Chemotherapy,25000,10,No,Recovered,4


In [11]:
invalid_df.head(5)

""


In [20]:
import os
from dotenv import load_dotenv
import mssql_python

load_dotenv(override=True)

conn_string = os.getenv("CONNECTION_STRING")
conn = mssql_python.connect(conn_string)

if conn:
    print("Connection successful")
else:
    print("Connection failed")


Connection successful


In [13]:
cursor = conn.cursor()

In [14]:
stmt = "SELECT DB_NAME()"

cursor.execute(stmt)

db_name = cursor.fetchone()[0]
print(f"Database name: {db_name}")

Database name: HospitalPatientDb


In [19]:
cursor = conn.cursor()

# create schema
stmt = """
IF SCHEMA_ID ('staging') IS NULL
BEGIN
    EXEC('CREATE SCHEMA staging')
    SELECT 'created';
END
ELSE
BEGIN
    SELECT 'exists';
END;
"""

try:
    cursor.execute(stmt)
    result = cursor.fetchone()[0]
    conn.commit()

    if result == 'created':
        print("Schema created successfully")
    elif result == 'exists':
        print("Schema already exists")
except Exception as e:
    conn.rollback()
    print("Failed to create schema")
    print(e)

Schema already exists


In [21]:
# create a table
stmt = """
IF NOT EXISTS (
    SELECT 1 FROM sys.tables
    WHERE name = 'Patient'
    AND schema_id = SCHEMA_ID('staging')
)
BEGIN
    CREATE TABLE staging.Patient (
        PatientId INT NOT NULL PRIMARY KEY,
        Age INT NOT NULL,
        Gender VARCHAR(10) NOT NULL,
        Condition VARCHAR(50) NOT NULL,
        ProcedureName VARCHAR(50) NOT NULL,
        Cost decimal(18,2) NOT NULL,
        LengthOfStay INT NOT NULL,
        Readmission VARCHAR(10) NOT NULL,
        Outcome VARCHAR(30) NOT NULL,
        Satisfaction INT NOT NULL
    );
    SELECT 'created';
END
ELSE
BEGIN
    SELECT 'exists';
END;
"""

try:
    cursor.execute(stmt)
    result = cursor.fetchone()[0]
    conn.commit()

    if result == 'created':
        print("Table created successfully")
    elif result == 'exists':
        print("Table already exists")
except Exception as ex:
    conn.rollback()
    print("Failed to create table")
    print(ex)

Table already exists


In [17]:
stmt="""
INSERT INTO staging.Patient(
    PatientId, Age, Gender, Condition, ProcedureName, Cost, LengthOfStay, Readmission, Outcome, Satisfaction
    )
VALUES (?,?,?,?,?,?,?,?,?,?);
"""

try:
    for _, row in valid_df.iterrows():
        cursor.execute(stmt,
                       row["Patient_ID"],
                       row["Age"],
                       row["Gender"],
                       row["Condition"],
                       row["Procedure"],
                       row["Cost"],
                       row["Length_of_Stay"],
                       row["Readmission"],
                       row["Outcome"],
                       row["Satisfaction"])
        cursor.commit()
    print("Inserted successfully into staging.Patient")
except Exception as ex:
    conn.rollback()
    print("Failed to insert to staging.Patient table")
    print(ex)


Failed to insert to staging.Patient table
Driver Error: Integrity constraint violation; DDBC Error: [Microsoft][SQL Server]Violation of PRIMARY KEY constraint 'PK__Patient__970EC3668BFC24F1'. Cannot insert duplicate key in object 'staging.Patient'. The duplicate key value is (1).


In [23]:
stmt = """
SELECT * FROM staging.Patient;
"""

cursor.execute(stmt)

rows = cursor.fetchall()

print(f"Total rows: {len(rows)}")

if len(valid_df) == len(rows):
    print("All rows inserted successfully and counted")
else:
    print("Some rows not inserted successfully")

for x in range(0,10):
    print(rows[x])

Total rows: 984
All rows inserted successfully and counted
(1, 45, 'Female', 'Heart Disease', 'Angioplasty', 15000.00, 5, 'No', 'Recovered', 4)
(2, 60, 'Male', 'Diabetes', 'Insulin Therapy', 2000.00, 3, 'Yes', 'Stable', 3)
(3, 32, 'Female', 'Fractured Arm', 'X-Ray and Splint', 500.00, 1, 'No', 'Recovered', 5)
(4, 75, 'Male', 'Stroke', 'CT Scan and Medication', 10000.00, 7, 'Yes', 'Stable', 2)
(5, 50, 'Female', 'Cancer', 'Surgery and Chemotherapy', 25000.00, 10, 'No', 'Recovered', 4)
(6, 68, 'Male', 'Hypertension', 'Medication and Counseling', 1000.00, 2, 'No', 'Stable', 4)
(7, 55, 'Female', 'Appendicitis', 'Appendectomy', 8000.00, 4, 'No', 'Recovered', 3)
(8, 40, 'Male', 'Fractured Leg', 'Cast and Physical Therapy', 3000.00, 6, 'No', 'Recovered', 4)
(9, 70, 'Female', 'Heart Attack', 'Cardiac Catheterization', 18000.00, 8, 'Yes', 'Stable', 2)
(10, 25, 'Male', 'Allergic Reaction', 'Epinephrine Injection', 100.00, 1, 'No', 'Recovered', 5)
